[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/06_numerical_integration_quadrature/exercises.ipynb)

# Exercises — Topic 06: Numerical Integration (Quadrature)

20 fully solved problems in 4 levels: concept checks, foundational computations and derivations, AI/ML and physics applications, and challenge proofs.

## Level 0 — Concept Check

### Problem L0.1: Simpson's free extra degree

Simpson's rule fits a parabola through three points. Show by direct computation that it nevertheless integrates every **cubic** exactly, and that it fails on quartics. Explain the structural reason.

**Solution.**

Work on $[-h, h]$ with midpoint $0$; the rule is $Q[f] = \frac{h}{3}\bigl(f(-h) + 4f(0) + f(h)\bigr)$.

**Cubics.** By linearity it suffices to check monomials.

| $f$ | $I = \int_{-h}^{h}f$ | $Q$ | equal? |
| :--- | :--- | :--- | :--- |
| $1$ | $2h$ | $\frac{h}{3}(1+4+1) = 2h$ | ✔ |
| $x$ | $0$ | $\frac{h}{3}(-h + 0 + h) = 0$ | ✔ |
| $x^{2}$ | $\frac{2h^{3}}{3}$ | $\frac{h}{3}(h^2 + 0 + h^2) = \frac{2h^{3}}{3}$ | ✔ |
| $x^{3}$ | $0$ | $\frac{h}{3}(-h^{3} + 0 + h^{3}) = 0$ | ✔ |
| $x^{4}$ | $\frac{2h^{5}}{5}$ | $\frac{h}{3}(h^{4}+0+h^{4}) = \frac{2h^{5}}{3}$ | ✘ |

So the degree of exactness is exactly $3$.

**Structural reason.** Exactness on $\mathbb{P}_2$ is automatic (the rule integrates the exact quadratic interpolant). The cubic monomial $x^{3}$ is **odd** about the midpoint, so its exact integral vanishes by symmetry — and the rule's nodes and weights are also symmetric, so its estimate vanishes too. Both sides are zero for free.

**The general principle.** *A symmetric rule with an odd number of nodes gains one degree of exactness.* It is the same parity argument that makes the central difference second-order in Topic 05 and the midpoint rule second-order in Theorem 2.

**Quantitatively.** The failure on $x^4$ gives the error constant directly: $E[x^{4}] = \frac{2h^5}{5} - \frac{2h^5}{3} = -\frac{4h^{5}}{15}$, so $E[f] = \frac{f^{(4)}(\xi)}{4!}\left(-\frac{4h^{5}}{15}\right) = -\frac{h^{5}}{90}f^{(4)}(\xi)$.

$$
\boxed{\text{Simpson has degree of exactness } 3 \text{ (not 2); error } -\tfrac{h^{5}}{90}f^{(4)}(\xi) \text{ on a panel of half-width } h}
$$

*Key takeaway:* Symmetry is worth a whole order — check the odd monomials before assuming a rule is only as good as the polynomial it fits.

### Problem L0.2: Why nobody uses high-order Newton–Cotes

Explain what goes wrong with closed Newton–Cotes rules as the order grows, give the numerical evidence, and state the correct way to increase accuracy.

**Solution.**

**The mechanism.** A Newton–Cotes rule integrates the polynomial interpolant through **equispaced** nodes. By Topic 04, that interpolant suffers the Runge phenomenon: the Lagrange basis functions $L_i$ oscillate with exponentially growing amplitude near the interval ends. Since $w_i = \int_a^b L_i$, the weights inherit the oscillation — they change sign and grow without bound.

**Numerical evidence.** Normalizing so that $\sum_i w_i = 1$ on $[0,1]$:

| $n$ (order) | smallest weight | $\sum_i \lvert w_i \rvert$ |
| :--- | :--- | :--- |
| 6 | $+0.032$ | $1.000$ |
| 7 | $+0.043$ | $1.000$ |
| 8 | $-0.160$ | $1.451$ |
| 10 | $-0.435$ | $3.065$ |
| 12 | $-1.392$ | $7.532$ |
| 14 | $-3.358$ | $20.344$ |

Negative weights first appear at $n = 8$, and $\sum\lvert w_i\rvert$ then grows geometrically.

**Two consequences.**
1. **Noise amplification.** If each sample carries error $\le \delta$, the rule's error is up to $\delta\sum_i\lvert w_i\rvert$. At $n = 14$ that is $20\delta$; at $n = 30$ it is astronomical. The rule is numerically unstable.
2. **Divergence.** Since $\sum\lvert w_i\rvert \to \infty$, the Pólya–Steklov theorem denies convergence for all continuous $f$; explicit divergent examples (e.g. $1/(1+25x^2)$) exist.

**What to do instead.**
- **Subdivide**: composite trapezoid/Simpson on many small panels. Global error $O(h^{2})$ / $O(h^{4})$, all weights positive, unconditionally stable.
- **Move the nodes**: Gauss (roots of orthogonal polynomials) or Clenshaw–Curtis (Chebyshev points). Both have **all-positive weights at every order**, hence stability and convergence for all continuous $f$ (Theorem 6).

$$
\boxed{\text{Newton–Cotes weights turn negative at } n = 8; \ \textstyle\sum\lvert w_i\rvert \to \infty \Rightarrow \text{instability and divergence}}
$$

*Key takeaway:* Positive weights are the stability certificate of a quadrature rule; raise accuracy by subdividing or by moving the nodes, never by raising the equispaced order.

### Problem L0.3: Local order versus global order

The single-panel Simpson error is $O(h^{5})$, yet the composite Simpson rule is only $O(h^{4})$. Where does the missing power go? State the general rule and check it on the trapezoid.

**Solution.**

**The accounting.** A composite rule on $[a,b]$ with panel width $h$ uses

$$
N = \frac{b-a}{h} \quad\text{panels}.
$$

Each panel contributes a *local* error $O(h^{k+1})$, and the errors add:

$$
E_{\text{global}} = \sum_{j=1}^{N} O(h^{k+1}) = N\cdot O(h^{k+1}) = \frac{b-a}{h}\cdot O(h^{k+1}) = O(h^{k}).
$$

**One power of $h$ is always lost**, and it is lost to the *number of panels*, which grows like $1/h$.

**Simpson.** Local $-\frac{(2h)^{5}}{2880}f^{(4)} = -\frac{h^{5}}{90}f^{(4)}$ on a panel of width $2h$; with $N/2$ such panels,

$$
E = -\frac{h^{5}}{90}\cdot\frac{N}{2}f^{(4)}(\xi) = -\frac{(b-a)h^{4}}{180}f^{(4)}(\xi) = O(h^{4}). \ ✔
$$

**Trapezoid.** Local $-\frac{h^{3}}{12}f''$; with $N = (b-a)/h$ panels,

$$
E = -\frac{h^{3}}{12}\cdot\frac{b-a}{h}f''(\xi) = -\frac{(b-a)h^{2}}{12}f''(\xi) = O(h^{2}). \ ✔
$$

**In terms of work.** With $N \propto 1/h$ function evaluations: composite trapezoid is $O(N^{-2})$, composite Simpson $O(N^{-4})$. Verified on $\int_0^1 e^{x}dx = 1.718281828$: composite Simpson errors are $5.79\times10^{-4}$ ($N=2$), $3.70\times10^{-5}$ ($N=4$), $2.33\times10^{-6}$ ($N=8$) — ratios $15.65$ and $15.91$, converging to $16 = 2^{4}$. ✔

$$
\boxed{\text{local } O(h^{k+1}) \times O(1/h) \text{ panels} = \text{global } O(h^{k})}
$$

*Key takeaway:* Always quote composite rules by their *global* order; the one-power loss is structural and identical to the local/global truncation-error relationship for ODE solvers.

### Problem L0.4: Choosing a quadrature method

Pick a method and justify: (a) $\int_0^1 e^{-x^{2}}dx$ to $10^{-14}$; (b) $\int_0^1 \frac{\log x}{\sqrt{x}}dx$; (c) $\int_0^{2\pi}\frac{dx}{2+\cos x}$; (d) a $50$-dimensional expectation under a trained normalizing flow.

**Solution.**

(a) **Gauss–Legendre (or Clenshaw–Curtis) with 10–20 nodes.** $e^{-x^2}$ is entire, so the Gauss error $\frac{f^{(2n)}(\xi)}{(2n)!}\int\omega^2$ decays *geometrically* in $n$. Twenty nodes reach machine precision; a composite Simpson rule would need thousands of evaluations for $10^{-14}$ and would be limited by roundoff accumulation.

(b) **A variable transformation, or Gauss–Jacobi.** The integrand has an integrable but unbounded singularity at $0$: no derivative bound exists, so all the error theorems are vacuous and Romberg stalls (Euler–Maclaurin needs $f\in C^{2m+2}$). Substituting $x = t^{2}$ gives $\int_0^1 \frac{2\log t}{t}\cdot 2t\,dt = 4\int_0^1 \log t\,dt$, which is still singular but only logarithmically; the professional answers are the **tanh–sinh (double-exponential)** transformation, which pushes endpoint singularities to where the weights vanish doubly exponentially, or `scipy.integrate.quad(..., weight='alg-log')`.

(c) **The plain composite trapezoid rule.** The integrand is smooth and **periodic**, so every Euler–Maclaurin term $\bigl[f^{(2k-1)}\bigr]_a^b$ vanishes and convergence is spectral. Measured: $N=8$ gives error $1.93\times10^{-4}$, $N=16$ gives $5.12\times10^{-9}$, and $N=32$ reaches $4.4\times10^{-16}$ — machine precision with 32 evaluations. Gauss–Legendre would be *worse* here.

(d) **Monte Carlo with the flow's own sampler.** At $d = 50$ any tensor grid is impossible ($3^{50} \approx 7\times10^{23}$ points), and the whole point of a normalizing flow is that it provides exact iid samples with tractable densities — so estimate the expectation by sampling, and reduce variance with control variates or antithetic pairs. Randomized QMC (scrambled Sobol' pushed through the flow) can help if the effective dimension is low.

$$
\boxed{\text{(a) Gauss–Legendre, (b) tanh–sinh / Gauss–Jacobi, (c) periodic trapezoid, (d) Monte Carlo}}
$$

*Key takeaway:* Match the rule to the integrand's *structure* — analyticity, endpoint behaviour, periodicity, and dimension — not to a default.

## Level 1 — Foundation

### Problem L1.1: Midpoint, trapezoid, Simpson on a single interval

Approximate $\int_0^1 e^{x}dx = e - 1 = 1.718281828459045$ with the one-panel midpoint, trapezoid, and Simpson rules. Verify each predicted error term and confirm the midpoint/trapezoid relationship.

**Solution.**

**Midpoint.** $M = 1\cdot e^{0.5} = 1.648721270700$, error $I - M = +6.9561\times10^{-2}$.

Predicted: $\frac{(b-a)^{3}}{24}f''(\xi) = \frac{e^{\xi}}{24}$ with $\xi\in(0,1)$, i.e. between $0.04167$ and $0.11326$ — and $0.069561$ lies in the band. ✔

**Trapezoid.** $T = \frac{1}{2}(e^{0} + e^{1}) = 1.859140914230$, error $I - T = -1.4086\times10^{-1}$.

Predicted: $-\frac{(b-a)^3}{12}f''(\xi) = -\frac{e^{\xi}}{12} \in (-0.2265, -0.0833)$. ✔

**The two-to-one relationship.** $\frac{\lvert I - T\rvert}{\lvert I - M\rvert} = \frac{0.140859}{0.069561} = 2.0250 \approx 2$, and the signs are opposite — exactly as Theorem 2 predicts ($\frac{1}{12}$ versus $\frac{1}{24}$, opposite signs). The true value is therefore **bracketed**: $M \lt I \lt T$.

**Simpson as the extrapolation.** Eliminating $f''$ between the two error formulas gives

$$
S = \frac{2M + T}{3} = \frac{2(1.648721270700) + 1.859140914230}{3} = 1.718861151877,
$$

with error $I - S = -5.7932\times10^{-4}$ — **120 times smaller** than the midpoint error, obtained by *combining* estimates already computed rather than by any new evaluation. And indeed $\frac{2M+T}{3} = \frac{b-a}{6}(f(a)+4f(m)+f(b))$ is Simpson's rule.

Predicted Simpson error: $-\frac{(b-a)^{5}}{2880}f^{(4)}(\xi) = -\frac{e^{\xi}}{2880}\in(-9.44\times10^{-4}, -3.47\times10^{-4})$. ✔

$$
\boxed{M = 1.648721,\ T = 1.859141,\ S = \tfrac{2M+T}{3} = 1.718861; \ \text{errors } 7.0\times10^{-2},\, 1.4\times10^{-1},\, 5.8\times10^{-4}}
$$

*Key takeaway:* Midpoint and trapezoid bracket the answer with errors in a $1{:}2$ ratio, and Simpson is precisely the Richardson combination that cancels the shared $f''$ term.

### Problem L1.2: How many panels for a target accuracy?

For $\int_0^1 e^{x}dx$ with tolerance $10^{-6}$, determine the number of panels required by the composite trapezoid and composite Simpson rules. Then verify the predicted convergence rates against measured errors.

**Solution.**

Here $b - a = 1$ and $\max_{[0,1]}\lvert f''\rvert = \max\lvert f^{(4)}\rvert = e = 2.718282$.

**Composite trapezoid.** $\lvert E\rvert \le \frac{(b-a)h^{2}}{12}e \le 10^{-6}$:

$$
h^{2} \le \frac{12\times10^{-6}}{e} = 4.4146\times10^{-6} \implies h \le 2.1011\times10^{-3} \implies N \ge 475.9,
$$

so $\boxed{N = 476}$ panels (477 evaluations).

**Composite Simpson.** $\lvert E\rvert \le \frac{(b-a)h^{4}}{180}e \le 10^{-6}$:

$$
h^{4} \le \frac{180\times10^{-6}}{e} = 6.6219\times10^{-5} \implies h \le 9.0208\times10^{-2} \implies N \ge 11.09,
$$

and $N$ must be even, so $N = 12$ panels (13 evaluations) — a **37-fold** saving.

**Measured errors** (exact $= 1.718281828459045$):

| $N$ | trapezoid error | ratio | $N$ | Simpson error | ratio |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | $1.4086\times10^{-1}$ | — | 2 | $5.7932\times10^{-4}$ | — |
| 2 | $3.5649\times10^{-2}$ | $3.951$ | 4 | $3.7013\times10^{-5}$ | $15.65$ |
| 4 | $8.9401\times10^{-3}$ | $3.988$ | 8 | $2.3262\times10^{-6}$ | $15.91$ |

Ratios converge to $4 = 2^{2}$ and $16 = 2^{4}$, confirming $O(h^{2})$ and $O(h^{4})$. ✔

**Sanity check of the bound.** At $N = 12$, Simpson's predicted bound is $\frac{(1/12)^{4}}{180}e = 9.0\times10^{-7} \lt 10^{-6}$ ✔, and extrapolating the measured sequence gives $\approx 2.3\times10^{-6}\times(8/12)^{4} = 4.6\times10^{-7}$ — comfortably inside.

$$
\boxed{N_{\text{trap}} = 476 \text{ vs } N_{\text{Simp}} = 12 \text{ for } 10^{-6}}
$$

*Key takeaway:* Two extra orders of accuracy convert hundreds of evaluations into a dozen — order dominates every other consideration for smooth integrands.

### Problem L1.3: A Romberg table

Build a Romberg table for $\int_0^1 e^{x}dx$ with $5$ levels, report the errors, and identify the observed order of each column.

**Solution.**

With $R_{k,0} = T(h_k)$, $h_k = 2^{-k}$, and $R_{k,m} = \frac{4^{m}R_{k,m-1} - R_{k-1,m-1}}{4^{m}-1}$:

| $k$ | $R_{k,0}$ | $R_{k,1}$ | $R_{k,2}$ | $R_{k,3}$ | $R_{k,4}$ |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 0 | $1.859140914230$ | | | | |
| 1 | $1.753931092465$ | $1.718861151877$ | | | |
| 2 | $1.727221904558$ | $1.718318841922$ | $1.718282687925$ | | |
| 3 | $1.720518592164$ | $1.718284154700$ | $1.718281842218$ | $1.718281828795$ | |
| 4 | $1.718841128580$ | $1.718281974052$ | $1.718281828675$ | $1.718281828460$ | $1.718281828459$ |

**Errors** ($R_{k,m} - I$, with $I = 1.718281828459045$):

| $k$ | col 0 | col 1 | col 2 | col 3 | col 4 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 0 | $+1.409\times10^{-1}$ | | | | |
| 1 | $+3.565\times10^{-2}$ | $+5.793\times10^{-4}$ | | | |
| 2 | $+8.940\times10^{-3}$ | $+3.701\times10^{-5}$ | $+8.595\times10^{-7}$ | | |
| 3 | $+2.237\times10^{-3}$ | $+2.326\times10^{-6}$ | $+1.376\times10^{-8}$ | $+3.355\times10^{-10}$ | |
| 4 | $+5.593\times10^{-4}$ | $+1.456\times10^{-7}$ | $+2.163\times10^{-10}$ | $+1.344\times10^{-12}$ | $+3.308\times10^{-14}$ |

**Observed orders** (error ratio between consecutive rows within a column):

- Column 0: $3.95, 3.99, 4.00, 4.00 \to 4 = 2^{2}$, order $2$ (trapezoid).
- Column 1: $15.65, 15.91, 15.98 \to 16 = 2^{4}$, order $4$ — and indeed column 1 *is* composite Simpson ($R_{1,1} = 1.718861151877$ matches the Simpson value from L1.1 exactly).
- Column 2: $62.5, 63.6 \to 64 = 2^{6}$, order $6$ (Boole's rule).
- Column 3: $\approx 250 \to 256 = 2^{8}$, order $8$.

**The payoff.** The last diagonal entry uses $2^{4}+1 = 17$ function evaluations and is accurate to $3.3\times10^{-14}$ — **14 digits**. The plain trapezoid rule with the same 17 evaluations gives only $5.6\times10^{-4}$: Romberg is $1.7\times10^{10}$ times more accurate at identical cost.

$$
\boxed{\text{Column } m \text{ has order } 2m+2; \ 17 \text{ evaluations give } 3.3\times10^{-14}}
$$

*Key takeaway:* Euler–Maclaurin's even-power expansion is what makes each Romberg column jump two orders; the trapezoid values are reused incrementally, so the table is essentially free.

### Problem L1.4: Deriving two-point Gauss–Legendre from scratch

Find the nodes and weights of the two-point rule on $[-1,1]$ that is exact for all polynomials of degree $\le 3$. Verify the answer two ways: by the moment equations and by orthogonality.

**Solution.**

**Method 1 — moment equations.** Require $Q[x^{j}] = \int_{-1}^{1}x^{j}dx$ for $j = 0,1,2,3$:

$$
w_1 + w_2 = 2, \qquad w_1x_1 + w_2x_2 = 0, \qquad w_1x_1^{2} + w_2x_2^{2} = \frac23, \qquad w_1x_1^{3} + w_2x_2^{3} = 0 .
$$

Four equations, four unknowns. Try the symmetric ansatz $x_2 = -x_1 = t \gt 0$, $w_1 = w_2 = w$ (justified below). Equations 2 and 4 are then satisfied identically by oddness. Equation 1 gives $w = 1$; equation 3 gives $2t^{2} = \tfrac23$, so $t = 1/\sqrt3$:

$$
\int_{-1}^{1}f(x)dx \approx f\!\left(-\tfrac{1}{\sqrt3}\right) + f\!\left(\tfrac{1}{\sqrt3}\right), \qquad \tfrac{1}{\sqrt3} = 0.5773502692 .
$$

*Why symmetry is forced:* the weight function $\rho \equiv 1$ is even on a symmetric interval, so the orthogonal polynomials have definite parity and their roots come in $\pm$ pairs.

**Method 2 — orthogonality (the general recipe).** The nodes must be the roots of the degree-2 Legendre polynomial. Build it by Gram–Schmidt against $\rho \equiv 1$ on $[-1,1]$: $p_0 = 1$, $p_1 = x$, and

$$
p_2 = x^{2} - \frac{\langle x^{2}, 1\rangle}{\langle 1,1\rangle} - \frac{\langle x^{2},x\rangle}{\langle x,x\rangle}x = x^{2} - \frac{2/3}{2} - 0 = x^{2} - \frac13 ,
$$

whose roots are $x = \pm 1/\sqrt3$. ✔ The weights are then interpolatory: $w_i = \int_{-1}^{1}L_i\,dx$, giving $w_1 = w_2 = 1$.

**Verification of degree 3.** $Q[1] = 2 = I$ ✔; $Q[x] = 0 = I$ ✔; $Q[x^{2}] = \tfrac13+\tfrac13 = \tfrac23 = I$ ✔; $Q[x^{3}] = 0 = I$ ✔; $Q[x^{4}] = \tfrac19+\tfrac19 = \tfrac29$ versus $I = \tfrac25$ ✘. Degree of exactness exactly $3 = 2n-1$ with $n = 2$. ✔

**Application.** For $\int_0^1 e^{x}dx$, map $x = \frac{t+1}{2}$, $dx = \frac{dt}{2}$:

$$
Q = \frac12\left[ e^{(1 - 1/\sqrt3)/2} + e^{(1+1/\sqrt3)/2} \right] = 1.717896378008, \quad \text{error } 3.85\times10^{-4}.
$$

Compare Simpson (three evaluations): error $5.79\times10^{-4}$. **Two Gauss points beat three Simpson points.** With three Gauss points ($0, \pm\sqrt{3/5}$; weights $\tfrac89, \tfrac59, \tfrac59$) the error drops to $8.24\times10^{-7}$ — better than composite Simpson with nine evaluations ($2.33\times10^{-6}$).

$$
\boxed{x = \pm\tfrac{1}{\sqrt3},\ w = 1,\ 1; \text{ degree of exactness } 3}
$$

*Key takeaway:* Freeing the nodes doubles the degree of exactness for the same evaluation count — the single biggest structural gain available in quadrature.

### Problem L1.5: Spectral accuracy of the periodic trapezoid rule

Evaluate $\int_0^{2\pi}\frac{dx}{2 + \cos x} = \frac{2\pi}{\sqrt3}$ by the composite trapezoid rule with $N = 4, 8, 16, 32$ points. Explain the observed convergence rate.

**Solution.**

**Rule.** For a $2\pi$-periodic $f$, the endpoints coincide, so the composite trapezoid rule is simply the average of $N$ equispaced samples:

$$
T_N = \frac{2\pi}{N}\sum_{j=0}^{N-1} f\!\left( \frac{2\pi j}{N} \right).
$$

**Results** (exact $= 2\pi/\sqrt3 = 3.627598728468436$):

| $N$ | $T_N$ | error |
| :--- | :--- | :--- |
| 4 | $3.665191429188$ | $3.759\times10^{-2}$ |
| 8 | $3.627791516645$ | $1.928\times10^{-4}$ |
| 16 | $3.627598733591$ | $5.123\times10^{-9}$ |
| 32 | $3.627598728468$ | $4.4\times10^{-16}$ |

The error is **squared** each time $N$ doubles. That is not $O(h^{2})$ or even $O(h^{10})$ — it is *geometric*: $E \approx C\rho^{-N}$.

**Why.** Euler–Maclaurin (Theorem 4) gives

$$
T(h) - I = \sum_{k\ge1}\frac{B_{2k}}{(2k)!}h^{2k}\left[ f^{(2k-1)}(2\pi) - f^{(2k-1)}(0) \right] + \cdots
$$

For a periodic $f$ **every bracket vanishes**: $f^{(2k-1)}(2\pi) = f^{(2k-1)}(0)$. So the entire asymptotic expansion is identically zero and the error decays faster than any power of $h$.

**The exact rate.** For $f = 1/(2+\cos x)$, the Fourier-analytic error formula is $E = \sum_{m\neq0}\hat f(mN)$, and $\hat f(k) = \frac{2\pi}{\sqrt3}\rho^{-\lvert k\rvert}$ with $\rho = 2+\sqrt3 = 3.7320508$ (the pole of the analytic continuation, at $\cos z = -2$). Hence

$$
E \approx \frac{4\pi}{\sqrt3}\rho^{-N} = 7.2552\,\rho^{-N} .
$$

Check: $N = 8$ gives $7.2552\times3.7321^{-8} = 1.93\times10^{-4}$ ✔ (measured $1.928\times10^{-4}$); $N = 16$ gives $5.14\times10^{-9}$ ✔ (measured $5.123\times10^{-9}$). Agreement to three digits.

**Consequences.** The DFT/FFT is the periodic trapezoid rule, and this theorem is why Fourier coefficients of smooth periodic functions are computed to machine precision from a few dozen samples. It is also why spectral methods for periodic PDEs converge exponentially, and why one should *never* use Gauss–Legendre on a periodic integrand — the humble trapezoid rule is optimal there.

$$
\boxed{E \approx \tfrac{4\pi}{\sqrt3}(2+\sqrt3)^{-N}: \text{ geometric, machine precision at } N = 32}
$$

*Key takeaway:* Periodicity annihilates every Euler–Maclaurin term, turning the lowest-order rule in the book into a spectrally accurate method.

### Problem L1.6: Gauss versus composite Simpson, per evaluation

Compare Gauss–Legendre and composite Simpson on $\int_0^1 e^{x}dx$ at equal evaluation counts, and explain the difference in convergence character.

**Solution.**

**Measured errors** (exact $1.718281828459045$):

| Evaluations | composite Simpson | Gauss–Legendre |
| :--- | :--- | :--- |
| 2 | — | $3.855\times10^{-4}$ |
| 3 | $5.793\times10^{-4}$ | $8.241\times10^{-7}$ |
| 5 | $3.701\times10^{-5}$ | $6.537\times10^{-13}$ |
| 9 | $2.326\times10^{-6}$ | $2.2\times10^{-16}$ (machine precision) |

**Reading the table.** Two Gauss points already beat three Simpson points; three Gauss points beat nine Simpson points by a factor of three; by five Gauss points the answer is essentially exact.

**Why: algebraic versus geometric convergence.**
- Composite Simpson has error $-\frac{(b-a)h^{4}}{180}f^{(4)}(\xi) = O(N^{-4})$ — *algebraic*. Each extra digit costs a factor $10^{1/4} = 1.8$ in $N$.
- Gauss–Legendre with $n$ nodes has error $\frac{(b-a)^{2n+1}(n!)^{4}}{(2n+1)\left[(2n)!\right]^{3}}f^{(2n)}(\xi)$. For an **entire** function like $e^{x}$ the factorials in the denominator overwhelm everything and the error decays *super-geometrically*; for $f$ analytic in a Bernstein $\rho$-ellipse it decays like $\rho^{-2n}$ — *geometric*. Each extra digit costs a fixed *additive* number of nodes.

**When Simpson (or adaptivity) wins.**
1. $f$ has limited smoothness. If $f\in C^{4}$ but not $C^{6}$, Gauss's advantage evaporates: both rules converge at the same algebraic rate set by the smoothness.
2. $f$ has a singularity, a kink, or a narrow spike. Gauss nodes are fixed and global; adaptive subdivision concentrates effort where the trouble is.
3. Nodes must be **nested** for progressive refinement. Gauss nodes at $n$ and $n+1$ share nothing, so refining discards all previous work — the reason for Gauss–Kronrod extensions and for Clenshaw–Curtis, whose nodes *are* nested.

$$
\boxed{\text{Gauss: geometric in } n \text{ for analytic } f; \text{ composite Simpson: } O(N^{-4}) \text{ always}}
$$

*Key takeaway:* For smooth integrands, degree of exactness compounds into geometric convergence; for rough ones, adaptivity beats order every time.

## Level 2 — Applications in AI/ML & Physics

### Problem L2.1: Gauss–Hermite quadrature in Gaussian-process classification

A GP classifier with probit likelihood needs the predictive probability $\int \Phi(f)\,\mathcal{N}(f\mid\mu,\sigma^{2})\,df$. Derive the Gauss–Hermite form, evaluate it for $\mu = 1$, $\sigma = 1$ at several node counts, and compare with Monte Carlo.

**Solution.**

**Change of variables.** Gauss–Hermite is built for the weight $e^{-t^{2}}$ on $(-\infty,\infty)$. Substituting $f = \mu + \sqrt2\,\sigma t$ turns $\mathcal{N}(f\mid\mu,\sigma^2)df$ into $\frac{1}{\sqrt\pi}e^{-t^{2}}dt$, so

$$
\mathbb{E}_{\mathcal N(\mu,\sigma^{2})}\bigl[g(f)\bigr] = \frac{1}{\sqrt\pi}\int_{-\infty}^{\infty} g\bigl(\mu + \sqrt2\,\sigma t\bigr)e^{-t^{2}}dt \approx \frac{1}{\sqrt\pi}\sum_{i=1}^{n} w_i\, g\bigl(\mu + \sqrt2\,\sigma t_i\bigr),
$$

with $(t_i, w_i)$ from `numpy.polynomial.hermite.hermgauss` (or `scipy.special.roots_hermite`).

**Exact reference.** For the probit link there is a closed form: $\int\Phi(f)\mathcal N(f\mid\mu,\sigma^2)df = \Phi\!\left(\frac{\mu}{\sqrt{1+\sigma^{2}}}\right)$. With $\mu = \sigma = 1$: $\Phi(1/\sqrt2) = 0.760249938907$.

**Gauss–Hermite results.**

| $n$ | estimate | error |
| :--- | :--- | :--- |
| 3 | $0.765716781876$ | $5.47\times10^{-3}$ |
| 5 | $0.760535800517$ | $2.86\times10^{-4}$ |
| 10 | $0.760250621051$ | $6.82\times10^{-7}$ |
| 15 | $0.760249934124$ | $4.78\times10^{-9}$ |
| 20 | $0.760249938923$ | $1.60\times10^{-11}$ |

**Monte Carlo comparison** (RMSE over 400 replications, sampling $f\sim\mathcal N(1,1)$):

| $N$ | 10 | 100 | 1000 | 10000 |
| :--- | :--- | :--- | :--- | :--- |
| RMSE | $7.2\times10^{-2}$ | $2.4\times10^{-2}$ | $7.4\times10^{-3}$ | $2.4\times10^{-3}$ |

The RMSE falls by $\sqrt{10} = 3.16$ per decade of $N$ — the $O(N^{-1/2})$ law. **Ten Gauss–Hermite nodes are $3500\times$ more accurate than $10^{4}$ Monte Carlo samples**, using $1000\times$ fewer evaluations.

**Why quadrature wins here — and when it stops.** The integrand is smooth and **one-dimensional**, so the $2n-1$ exactness compounds geometrically. Real GP models need this integral for every test point and inside every EP site update, millions of times, so a 20-node deterministic rule with no sampling noise is exactly right. In $d$ latent dimensions the tensor rule costs $n^{d}$: at $d = 3$, $20^{3} = 8000$ is still fine; at $d = 10$ it is $10^{13}$ and one must switch to Monte Carlo or a factorized (mean-field) $q$ that reduces the problem back to $d$ one-dimensional integrals.

$$
\boxed{\mathbb{E}_{\mathcal N(\mu,\sigma^{2})}[g] \approx \tfrac{1}{\sqrt\pi}\sum_i w_i\,g(\mu + \sqrt2\sigma t_i); \; n=10 \text{ gives } 6.8\times10^{-7}}
$$

*Key takeaway:* Low-dimensional Gaussian expectations should be quadrature, not sampling — the transformation $f = \mu + \sqrt2\sigma t$ is the whole trick.

### Problem L2.2: Quadrature versus reparameterized sampling in the ELBO

A VAE maximizes $\mathcal{L} = \mathbb{E}_{q_\phi(z\mid x)}[\log p(x\mid z)] - \mathrm{KL}(q_\phi\Vert p)$. Explain which parts are quadrature problems, why the standard implementation uses a **single** Monte Carlo sample, and when Gauss–Hermite is preferable.

**Solution.**

**Decomposition.**
- The **KL term** for $q = \mathcal N(\mu,\mathrm{diag}\,\sigma^{2})$ and $p = \mathcal N(0,I)$ is a Gaussian integral with the closed form $\frac12\sum_j(\mu_j^{2} + \sigma_j^{2} - \log\sigma_j^{2} - 1)$. This is why that prior/posterior pairing is chosen: one of the two integrals disappears exactly.
- The **reconstruction term** $\mathbb{E}_{q}[\log p(x\mid z)]$ is a genuine $d_z$-dimensional integral against a Gaussian — a quadrature problem with $d_z$ typically $32$–$512$.

**Why one sample.** With the reparameterization $z = \mu + \sigma\odot\epsilon$, $\epsilon\sim\mathcal N(0,I)$, the estimator

$$
\hat{\mathcal L} = \log p\bigl(x\mid \mu + \sigma\odot\epsilon\bigr) - \mathrm{KL}
$$

is **unbiased** and its gradient is unbiased too (the randomness no longer depends on $\phi$). Three facts then make $N = 1$ optimal in practice:

1. **SGD only needs unbiasedness.** The optimizer already averages over minibatches and iterations; adding $N$ inner samples reduces the per-step variance by $1/N$ at $N\times$ the cost, whereas using the same compute for $N\times$ more *minibatches* reduces variance by $1/N$ **and** processes more data.
2. **Dimension.** At $d_z = 64$, a tensor Gauss–Hermite rule with even $3$ nodes per axis costs $3^{64}\approx 3\times10^{30}$ evaluations. Deterministic quadrature is not merely inefficient; it is impossible.
3. **The decoder is the cost.** $\log p(x\mid z)$ is a full neural-network forward pass, so evaluations are expensive and equal-cost comparisons favour breadth over depth.

**When Gauss–Hermite is preferable.**
- $d_z \le 3$, or a factorized likelihood that decomposes into **independent one-dimensional** Gaussian integrals — the situation in GP classification, in binary-likelihood latent-variable models, and in mean-field VI for GLMs. GPflow's `NDiagGHQuadrature` does exactly this with $20$ nodes per dimension.
- **Deterministic objectives are needed** for line search, for exact reproducibility, or for second-order optimizers, where Monte Carlo noise breaks the curvature estimates.
- **Small $\sigma$** (a confident posterior): the integrand is nearly linear over the effective support, so $3$–$5$ nodes give machine precision while sampling still pays the full $O(N^{-1/2})$.

**The general rule.** Deterministic quadrature when $d\le3$ *or* the integral factorizes into 1-D pieces; reparameterized Monte Carlo otherwise; and always prefer an analytic sub-integral (the KL term) to any numerical one.

$$
\boxed{\text{KL: closed form; reconstruction: 1-sample reparameterized MC (unbiased, } d_z \text{ large)}}
$$

*Key takeaway:* The ELBO is a quadrature problem whose dimension decides everything — and the reparameterization trick is what makes the $N=1$ Monte Carlo estimator usable as a *gradient*, not just a value.

### Problem L2.3: Sizing a Monte Carlo run, and the value of variance reduction

An expectation with $\sigma = 1$ must be estimated to RMSE $10^{-3}$, then $10^{-4}$. How many samples? Then quantify the saving from a control variate with correlation $\rho = 0.95$, and from antithetic variates on a monotone integrand.

**Solution.**

**Plain Monte Carlo.** $\mathrm{RMSE} = \sigma/\sqrt N$, so $N = (\sigma/\mathrm{RMSE})^{2}$:

$$
\mathrm{RMSE} = 10^{-3} \implies N = 10^{6}, \qquad \mathrm{RMSE} = 10^{-4} \implies N = 10^{8}.
$$

**One extra digit costs $100\times$ the work.** This single fact drives the entire variance-reduction literature — algorithmic improvement is far cheaper than brute force.

**Control variates.** Suppose $g$ is correlated with $f$ and $\mathbb{E}[g]$ is known exactly. Use

$$
\hat I = \frac{1}{N}\sum_k \Bigl[ f(X_k) - c\bigl(g(X_k) - \mathbb{E}[g]\bigr) \Bigr].
$$

Minimizing over $c$ gives $c^{\ast} = \frac{\operatorname{Cov}(f,g)}{\operatorname{Var}(g)}$ and

$$
\operatorname{Var}(\hat I) = \frac{\sigma^{2}(1-\rho^{2})}{N} .
$$

With $\rho = 0.95$: $1 - \rho^{2} = 0.0975$, so the variance drops by $10.3\times$ and the **required $N$ drops by $10.3\times$** — from $10^{6}$ to $97{,}500$ for RMSE $10^{-3}$. Equivalently, the RMSE at fixed $N$ improves by $\sqrt{0.0975} = 3.2\times$. Note the sharp threshold: $\rho = 0.9$ gives only $5.3\times$, $\rho = 0.99$ gives $50\times$ — control variates pay off dramatically only when the correlation is very high.

**Antithetic variates.** Pair $X_k$ with its reflection $\tilde X_k = 1 - X_k$ (uniform case) or $-\epsilon_k$ (Gaussian case) and average:

$$
\operatorname{Var}\!\left(\frac{f(X) + f(\tilde X)}{2}\right) = \frac{\sigma^{2}\bigl(1 + \operatorname{Corr}(f(X), f(\tilde X))\bigr)}{2}.
$$

For a **monotone** $f$ the two are negatively correlated, so the bracket is $\lt 1$ and the pair beats two independent samples; in the extreme of a linear $f$ the correlation is $-1$ and the variance is **zero** (the odd component is cancelled exactly). Cost: no extra evaluations of the random-number generator, and the same number of $f$ evaluations. This is why antithetic sampling is essentially free and always worth trying on smooth monotone integrands.

**The hierarchy.** Common random numbers $\lt$ antithetics $\lt$ control variates $\lt$ importance sampling $\lt$ stratification/QMC, in increasing order of effort and of potential gain.

$$
\boxed{N = 10^{6} \text{ then } 10^{8}; \ \rho = 0.95 \text{ control variate cuts } N \text{ by } 10.3\times}
$$

*Key takeaway:* At $O(N^{-1/2})$, accuracy is bought with quadratic work — so the correct engineering response to "not accurate enough" is variance reduction, not more samples.

### Problem L2.4: Quasi-Monte Carlo in ten dimensions

Estimate $\int_{[0,1]^{10}}\exp\!\left(\frac{1}{10}\sum_{i=1}^{10}x_i\right)dx$ (exact value $\left[10(e^{1/10}-1)\right]^{10} = 1.6556046996$) with plain Monte Carlo and with scrambled Sobol' points. Report the rates and explain the mechanism.

**Solution.**

**Measured mean absolute errors** (average over 20 independent replications):

| $N$ | Monte Carlo | scrambled Sobol' (RQMC) | ratio |
| :--- | :--- | :--- | :--- |
| $256$ | $7.43\times10^{-3}$ | $6.70\times10^{-5}$ | $111$ |
| $1024$ | $3.76\times10^{-3}$ | $6.88\times10^{-6}$ | $546$ |
| $4096$ | $1.50\times10^{-3}$ | $1.31\times10^{-6}$ | $1139$ |
| $16384$ | $8.93\times10^{-4}$ | $8.64\times10^{-8}$ | $10335$ |

**Rates.** Over the $64\times$ increase from $N = 256$ to $N = 16384$, Monte Carlo improved by $8.3\times \approx \sqrt{64} = 8$ — the $O(N^{-1/2})$ law, confirmed. RQMC improved by $775\times$, i.e. an empirical rate of $N^{-1.6}$: **better than $O(N^{-1})$**, which is typical for smooth, low-effective-dimension integrands where the scrambling contributes an extra $N^{-1/2}$ factor (Owen's theorem gives $O(N^{-3/2+\epsilon})$ for smooth $f$).

**Mechanism.** Random points clump and leave gaps; the fluctuation of the local point density is $O(\sqrt N)$ out of $N$, which is precisely the $N^{-1/2}$ error. A **low-discrepancy** sequence is constructed so that every axis-aligned box contains almost exactly its fair share of points. The Koksma–Hlawka inequality separates the two roles cleanly:

$$
\left\lvert \frac{1}{N}\sum_k f(u_k) - \int_{[0,1]^{d}}f \right\rvert \le \underbrace{V_{\mathrm{HK}}(f)}_{\text{integrand}}\ \cdot \underbrace{D_N^{\ast}(u_1,\ldots,u_N)}_{\text{point set}},
$$

and Sobol' points achieve $D_N^{\ast} = O\!\left(\frac{(\log N)^{d}}{N}\right)$.

**The two caveats.**
1. **$(\log N)^{d}$.** At $d = 10$ and $N = 16384$, $(\log N)^{10} \approx 9.7^{10} \approx 7\times10^{9}$ — the bound is vacuous. QMC nonetheless works, because what matters is the **effective dimension**: our integrand $\exp(\frac1{10}\sum x_i)$ is a sum of one-dimensional effects with no high-order interactions, so its ANOVA decomposition is concentrated in low-order terms. When strong interactions exist (or the integrand is discontinuous, making $V_{\mathrm{HK}} = \infty$), QMC degrades to MC.
2. **No error estimate from a single run.** QMC is deterministic, so there is no variance to report. **Scrambling** (random digit permutations, as in `scipy.stats.qmc.Sobol(scramble=True)`) restores unbiasedness and lets one run $R \approx 10$–$30$ independent replicates to obtain a genuine confidence interval — which is exactly how the table above was produced.

$$
\boxed{\text{At } N = 16384,\ d = 10: \text{ MC } 8.9\times10^{-4} \text{ vs RQMC } 8.6\times10^{-8}, \text{ a } 10^{4}\times \text{ gain}}
$$

*Key takeaway:* Replacing randomness by uniformity buys almost a full extra power of $N$ — provided the integrand is smooth and its effective dimension is low.

### Problem L2.5: Where the curse of dimensionality bites

For a tensor-product rule of order $k$ with $m$ points per axis in $d$ dimensions, derive the convergence rate in terms of total cost $N$, find the dimension at which Monte Carlo takes over, and give concrete point counts.

**Solution.**

**Cost and rate.** A tensor grid uses $N = m^{d}$ evaluations. A composite rule of order $k$ in each axis has error $O(m^{-k})$ per axis, and the errors add across axes without changing the order, so

$$
E = O\!\left(m^{-k}\right) = O\!\left( \bigl(N^{1/d}\bigr)^{-k} \right) = O\!\left(N^{-k/d}\right).
$$

**The crossover.** Monte Carlo achieves $O(N^{-1/2})$ regardless of $d$. Setting $\frac{k}{d} = \frac12$:

$$
d^{\ast} = 2k .
$$

| Rule | order $k$ | Monte Carlo wins for |
| :--- | :--- | :--- |
| Trapezoid / midpoint | 2 | $d \gt 4$ |
| Simpson | 4 | $d \gt 8$ |
| Boole | 6 | $d \gt 12$ |

Beyond $d^{\ast}$ the deterministic rate is *worse* than random sampling, and it keeps deteriorating: at $d = 100$ a tensor Simpson grid converges as $N^{-0.04}$ — to gain one digit you would multiply the work by $10^{25}$.

**Concrete point counts** (a very coarse $m$ per axis):

| $d$ | $m = 3$ | $m = 5$ | $m = 10$ |
| :--- | :--- | :--- | :--- |
| 5 | $243$ | $3125$ | $10^{5}$ |
| 10 | $59{,}049$ | $9.77\times10^{6}$ | $10^{10}$ |
| 20 | $3.49\times10^{9}$ | $9.54\times10^{13}$ | $10^{20}$ |
| 50 | $7.18\times10^{23}$ | $8.88\times10^{34}$ | $10^{50}$ |
| 100 | $5.15\times10^{47}$ | $7.89\times10^{69}$ | $10^{100}$ |

For reference, $10^{50}$ exceeds the number of atoms in the Earth ($\approx 10^{50}$) and $5\times10^{47}$ is already beyond any conceivable computation. Meanwhile $N = 10^{6}$ Monte Carlo samples give RMSE $\sigma\times10^{-3}$ at $d = 100$ just as at $d = 1$.

**The middle ground: sparse (Smolyak) grids.** Take a signed combination of tensor rules whose total index is bounded rather than the full product. The point count becomes $O\!\left(m(\log m)^{d-1}\right)$ and the error $O\!\left(m^{-k}(\log m)^{(d-1)(k+1)}\right)$ — a genuine improvement that buys perhaps $d \sim 10$–$30$ before Monte Carlo is forced. This is the workhorse of uncertainty quantification and polynomial-chaos expansions.

**Why ML lives here.** Latent spaces ($d_z \sim 64$), parameter posteriors ($d \sim 10^{6}$–$10^{9}$), and path integrals over trajectories are all far past $d^{\ast}$. Every expectation in modern machine learning is estimated by sampling, and every "trick" (reparameterization, control variates, importance weighting, MCMC) exists to reduce the constant $\sigma$ in $\sigma/\sqrt N$, because the *rate* cannot be improved without structure.

$$
\boxed{\text{tensor rule: } O(N^{-k/d}); \text{ MC wins for } d \gt 2k; \ 3^{100} = 5\times10^{47} \text{ points}}
$$

*Key takeaway:* Dimension converts a fast algebraic rate into a hopeless one; Monte Carlo's virtue is not its speed but its *indifference* to $d$.

### Problem L2.6: Expected calibration error is a composite midpoint rule

ECE with $M$ equal-width confidence bins estimates $\int_0^1 \lvert \mathrm{acc}(p) - p\rvert\,\pi(p)\,dp$. Identify the quadrature rule, decompose the error into discretization bias and sampling noise, and derive the optimal bin count for $n$ samples.

**Solution.**

**The rule.** Partitioning $[0,1]$ into $M$ bins of width $h = 1/M$ and evaluating the integrand once per bin (at the bin's average confidence, using the bin's empirical accuracy) is precisely a **composite midpoint rule** with panel weights $\hat\pi_j = n_j/n$ estimated from the data:

$$
\widehat{\mathrm{ECE}} = \sum_{j=1}^{M}\frac{n_j}{n}\Bigl\lvert \mathrm{acc}(B_j) - \mathrm{conf}(B_j) \Bigr\rvert .
$$

**Error source 1 — discretization bias.** By Theorem 3 the composite midpoint error is $\frac{(b-a)h^{2}}{24}g''(\xi) = O(M^{-2})$, where $g(p) = \lvert\mathrm{acc}(p)-p\rvert\pi(p)$. Within a bin, variation of the true gap is averaged away, so binning **underestimates** the true ECE, and the bias shrinks as $M$ grows:

$$
\mathrm{bias}_{\text{disc}} = O\!\left(M^{-2}\right).
$$

(Strictly, the absolute value creates a kink wherever $\mathrm{acc}(p) = p$, degrading the local order to $O(h^{2})$ overall but $O(h)$ in the kinked panel — the conclusion below is unchanged.)

**Error source 2 — sampling noise.** Each bin holds $n_j \approx n/M$ samples, so its empirical accuracy has standard deviation $\approx \frac{1}{2}\sqrt{M/n}$. Because the outer $\lvert\cdot\rvert$ is convex, Jensen's inequality makes that noise a **positive** bias: $\mathbb{E}\lvert X\rvert \ge \lvert\mathbb{E}X\rvert$. Averaging over bins,

$$
\mathrm{bias}_{\text{noise}} \;\approx\; c\,\sqrt{\frac{M}{n}} \quad (c \approx 0.4) .
$$

So ECE is biased **downward** by coarse binning and **upward** by fine binning — the two errors pull in opposite directions.

**Optimal bin count.** Balance the two:

$$
M^{-2} \sim \sqrt{\frac{M}{n}} \implies M^{-2}\sqrt n = \sqrt M \implies M^{5/2} = \sqrt n \implies \boxed{M^{\ast} \sim n^{1/5}} .
$$

| $n$ | $M^{\ast} = n^{1/5}$ |
| :--- | :--- |
| $10^{3}$ | $4.0$ |
| $10^{4}$ | $6.3$ |
| $10^{5}$ | $10.0$ |
| $10^{6}$ | $15.8$ |

This recovers the folklore default of $M = 10$–$15$ bins, and explains why it is a *default for a particular dataset size*: with $n = 1000$ validation points, $15$ bins is already over-binned and the reported ECE is inflated by sampling noise.

**Practical corollaries.** (i) Never compare ECE values computed with different $M$ or different $n$. (ii) Equal-**mass** bins (adaptive ECE) equalize $n_j$ and thus equalize the per-bin noise — a stratified quadrature rule, strictly better than equal width. (iii) Debiased and kernel-smoothed estimators of ECE are exactly the "regularized quadrature" answer to the same trade-off.

*Key takeaway:* ECE's notorious sensitivity to bin count is the classic quadrature trade-off — discretization bias $O(h^{2})$ against estimation noise $O(\sqrt{1/(nh)})$ — with the familiar $n^{1/5}$ balance point.

## Level 3 — Challenge

### Problem L3.1: Deriving the three-point Gauss–Legendre rule

Construct the $3$-point Gauss–Legendre rule on $[-1,1]$ from orthogonality, prove it has degree of exactness $5$, and derive its error constant. Then verify on $\int_0^1 e^{x}dx$.

**Solution.**

**Step 1 — the orthogonal polynomial.** The nodes must be the roots of the degree-3 Legendre polynomial for $\rho\equiv1$ on $[-1,1]$. Build it by Gram–Schmidt on $\{1, x, x^{2}, x^{3}\}$, or use parity: $p_3$ is odd, so $p_3 = x^{3} + ax$, and orthogonality to $x$ requires

$$
\int_{-1}^{1}(x^{3}+ax)x\,dx = \frac25 + \frac{2a}{3} = 0 \implies a = -\frac35 .
$$

(Orthogonality to $1$ and $x^{2}$ is automatic by parity.) So $p_3 = x^{3} - \tfrac35 x = x\left(x^{2}-\tfrac35\right)$ and the nodes are

$$
x_1 = -\sqrt{\tfrac35} = -0.7745966692, \quad x_2 = 0, \quad x_3 = +\sqrt{\tfrac35} = 0.7745966692 .
$$

**Step 2 — the weights.** Interpolatory: $w_i = \int_{-1}^{1}L_i\,dx$. By symmetry $w_1 = w_3 = w$ and $w_2 = v$, and exactness on $1$ and $x^{2}$ gives

$$
2w + v = 2, \qquad 2w\cdot\frac35 = \frac23 \implies w = \frac59, \quad v = 2 - \frac{10}{9} = \frac89 .
$$

$$
\int_{-1}^{1}f \approx \frac59 f\!\left(-\sqrt{\tfrac35}\right) + \frac89 f(0) + \frac59 f\!\left(\sqrt{\tfrac35}\right).
$$

All weights positive, summing to $2 = \int_{-1}^{1}dx$. ✔

**Step 3 — degree of exactness is exactly $5 = 2n-1$.**

| $f$ | $I$ | $Q$ |
| :--- | :--- | :--- |
| $1$ | $2$ | $\tfrac59+\tfrac89+\tfrac59 = 2$ ✔ |
| $x^{2}$ | $2/3$ | $2\cdot\tfrac59\cdot\tfrac35 = \tfrac23$ ✔ |
| $x^{4}$ | $2/5$ | $2\cdot\tfrac59\cdot\tfrac9{25} = \tfrac25$ ✔ |
| $x^{6}$ | $2/7$ | $2\cdot\tfrac59\cdot\tfrac{27}{125} = \tfrac{6}{25} = 0.24$ ✘ ($2/7 = 0.2857$) |

Odd powers vanish on both sides by symmetry. So exactness holds through degree $5$ and fails at $6$. ✔

**Step 4 — the error constant.** By Theorem 5 with $n = 3$ on $[-1,1]$ (so $b-a = 2$):

$$
E = \frac{(b-a)^{2n+1}(n!)^{4}}{(2n+1)\bigl[(2n)!\bigr]^{3}}f^{(6)}(\xi) = \frac{2^{7}\,(6)^{4}}{7\,(720)^{3}}f^{(6)}(\xi) = \frac{128\times1296}{7\times3.73248\times10^{8}}f^{(6)}(\xi) = \frac{f^{(6)}(\xi)}{15750}.
$$

Cross-check with Step 3: exactness fails first on $x^{6}$, where $f^{(6)} = 720$ and $E = I - Q = \tfrac27 - \tfrac6{25} = \tfrac{50 - 42}{175} = \tfrac{8}{175}$; the formula gives $\tfrac{720}{15750} = \tfrac{8}{175}$ ✔ — exact agreement.

**Step 5 — numerical verification.** Mapping to $[0,1]$ via $x = \frac{t+1}{2}$:

$$
Q = \frac12\left[ \tfrac59 e^{(1-\sqrt{3/5})/2} + \tfrac89 e^{1/2} + \tfrac59 e^{(1+\sqrt{3/5})/2} \right] = 1.718281004373,
$$

error $8.24\times10^{-7}$ from **three** evaluations. Composite Simpson needs nine evaluations to reach $2.33\times10^{-6}$ — still worse.

Predicted error: the general formula with $b - a = 1$ and $n = 3$ gives $\frac{(1)^{7}(3!)^{4}}{7\,(720)^{3}}f^{(6)}(\xi) = \frac{e^{\xi}}{2\,016\,000}$, which for $\xi\in(0,1)$ lies in $\bigl(4.96\times10^{-7},\ 1.35\times10^{-6}\bigr)$ — and the measured $8.24\times10^{-7}$ sits inside that band. ✔

$$
\boxed{x = 0, \pm\sqrt{3/5};\ w = \tfrac89, \tfrac59, \tfrac59;\ \text{degree } 5;\ E = \tfrac{f^{(6)}(\xi)}{15750} \text{ on } [-1,1]}
$$

*Key takeaway:* Parity plus two moment conditions determine a Gauss rule completely, and the same monomial check that establishes the degree also pins down the error constant.

### Problem L3.2: Euler–Maclaurin, Romberg orders, and periodic magic

Prove that the composite trapezoid error admits an expansion in even powers of $h$, deduce that Romberg column $m$ has order $2m+2$, and prove that the expansion vanishes identically for smooth periodic integrands.

**Solution.**

**Part 1 — the Euler–Maclaurin expansion.** On a single panel $[0,h]$, integrate by parts repeatedly using the Bernoulli polynomials $B_k(t)$, which satisfy $B_k'(t) = kB_{k-1}(t)$, $B_0 = 1$, $B_1(t) = t - \tfrac12$, and $B_k(0) = B_k(1) = B_k$ (the Bernoulli numbers) for $k \ge 2$ even, with $B_k = 0$ for $k \ge 3$ odd. Substituting $x = h t$:

$$
\int_0^{h}f\,dx = h\int_0^1 f(ht)\,dt, \qquad \int_0^1 g(t)\,dt = \frac{g(0)+g(1)}{2} - \sum_{k=1}^{m}\frac{B_{2k}}{(2k)!}\Bigl[g^{(2k-1)}\Bigr]_0^1 + R_m .
$$

The first integration by parts with $B_1(t) = t - \tfrac12$ produces the trapezoid term; each subsequent pair of integrations produces one $B_{2k}$ term, and the odd-index terms drop out because $B_{2k+1} = 0$ for $k \ge 1$.

Summing over the $N$ panels of the composite rule, all **interior** boundary terms telescope — panel $j$'s right-endpoint contribution cancels panel $j+1$'s left-endpoint contribution — leaving only the outer endpoints:

$$
T(h) - \int_a^b f = \sum_{k=1}^{m}\frac{B_{2k}}{(2k)!}h^{2k}\Bigl[ f^{(2k-1)}(b) - f^{(2k-1)}(a) \Bigr] + O(h^{2m+2}). \qquad \blacksquare
$$

The leading term is $\frac{h^{2}}{12}\bigl[f'(b)-f'(a)\bigr]$, consistent with $B_2 = \tfrac16$ and with Theorem 3's $-\frac{(b-a)h^{2}}{12}f''(\xi)$ (the two agree by the mean value theorem applied to $f'(b)-f'(a) = (b-a)f''(\xi)$, up to sign convention).

**Part 2 — Romberg column $m$ has order $2m+2$.** The expansion contains **only even powers**, so in Richardson's notation $p = 2$ and $q = 2$. Induct: suppose $R_{k,m-1} = I + \sum_{i\ge m}c_i^{(m-1)}h_k^{2i}$. Since $h_k = h_{k-1}/2$, we have $h_k^{2i} = 4^{-i}h_{k-1}^{2i}$ and

$$
4^{m}R_{k,m-1} - R_{k-1,m-1} = (4^{m}-1)I + \sum_{i\ge m}c_i^{(m-1)}h_{k-1}^{2i}\bigl(4^{m-i}-1\bigr),
$$

whose $i = m$ term vanishes exactly. Dividing by $4^{m}-1$ leaves $R_{k,m} = I + O(h^{2m+2})$. $\blacksquare$

Verified numerically on $\int_0^1 e^{x}dx$: the error ratios between consecutive rows are $4.00$ (column 0), $15.98$ (column 1), $63.6$ (column 2), $\approx256$ (column 3) — i.e. $2^{2}, 2^{4}, 2^{6}, 2^{8}$, orders $2,4,6,8$. ✔

**Part 3 — periodic integrands.** If $f$ is $C^{\infty}$ and $(b-a)$-periodic then $f^{(2k-1)}(b) = f^{(2k-1)}(a)$ for **every** $k$, so every bracket in the expansion is zero and

$$
T(h) - I = O(h^{2m+2}) \quad \text{for every } m .
$$

The error decays faster than any power of $h$. For $f$ analytic in a strip of half-width $\log\rho$ around the real axis, the Fourier-series argument sharpens this to a geometric rate: $T_N - I = \sum_{m\neq0}\hat f(mN)$ and $\lvert\hat f(k)\rvert \le C\rho^{-\lvert k\rvert}$, so $\lvert T_N - I\rvert \le 2C\rho^{-N}/(1-\rho^{-N})$.

**Verification.** For $\int_0^{2\pi}\frac{dx}{2+\cos x} = \frac{2\pi}{\sqrt3}$, the singularity of the analytic continuation is at $\cos z = -2$, giving $\rho = 2+\sqrt3 = 3.7320508$ and $C = \frac{2\pi}{\sqrt3}$. Predicted error $\approx\frac{4\pi}{\sqrt3}\rho^{-N}$:

| $N$ | predicted | measured |
| :--- | :--- | :--- |
| 8 | $1.93\times10^{-4}$ | $1.928\times10^{-4}$ |
| 16 | $5.14\times10^{-9}$ | $5.123\times10^{-9}$ |
| 32 | $3.6\times10^{-18}$ | $4.4\times10^{-16}$ (machine precision) |

Three-digit agreement at $N = 8$ and $16$; at $N = 32$ the true error is below the roundoff floor.

$$
\boxed{T(h)-I = \textstyle\sum_k \frac{B_{2k}}{(2k)!}h^{2k}\bigl[f^{(2k-1)}\bigr]_a^b;\ \text{periodic} \Rightarrow \text{all terms vanish}}
$$

*Key takeaway:* One theorem explains both Romberg's two-orders-per-column and the spectral accuracy of the trapezoid rule on periodic data — and the FFT's exponential accuracy is a corollary.

### Problem L3.3: Positive weights, stability, and convergence for all continuous functions

Prove that Gauss weights are positive, that positivity implies a noise bound independent of $n$, and that it implies convergence $Q_n[f]\to I[f]$ for every $f\in C[a,b]$. Contrast with Newton–Cotes.

**Solution.**

**Part 1 — positivity.** Fix $i$ and test the rule on $L_i^{2}$, where $L_i$ is the Lagrange basis polynomial for the Gauss nodes. Then $\deg L_i^{2} = 2(n-1) = 2n-2 \le 2n-1$, so the rule is exact:

$$
w_i = \sum_j w_j\,\delta_{ij} = \sum_j w_j L_i(x_j)^{2} = Q_n\bigl[L_i^{2}\bigr] = \int_a^b L_i(x)^{2}\rho(x)\,dx \gt 0,
$$

strict because $L_i^{2}\ge0$ is continuous, equals $1$ at $x_i$, and $\rho$ has positive mass. Exactness on $f\equiv1$ gives $\sum_i w_i = \int_a^b\rho\,dx =: \mu_0$. $\blacksquare$

**Part 2 — a noise bound independent of $n$.** If the computed samples satisfy $\lvert \tilde f(x_i) - f(x_i)\rvert \le \delta$, then

$$
\bigl\lvert Q_n[\tilde f] - Q_n[f] \bigr\rvert \le \sum_i \lvert w_i\rvert\,\delta = \left(\sum_i w_i\right)\delta = \mu_0\,\delta ,
$$

using positivity to replace $\sum\lvert w_i\rvert$ by $\sum w_i$. **The amplification factor is $\mu_0 = \int\rho$ for every $n$** — the rule is unconditionally stable; adding nodes never amplifies data error. $\blacksquare$

**Part 3 — Stieltjes' convergence theorem.** Let $f\in C[a,b]$ and $\epsilon \gt 0$. By Weierstrass there is $p\in\mathbb{P}_{m}$ with $\lVert f-p\rVert_\infty \lt \epsilon$. For every $n$ with $2n-1\ge m$ the rule is exact on $p$, so

$$
\bigl\lvert I[f] - Q_n[f]\bigr\rvert = \bigl\lvert I[f-p] + \underbrace{I[p]-Q_n[p]}_{=0} - Q_n[f-p] \bigr\rvert \le \lVert f-p\rVert_\infty\!\left(\int_a^b\rho + \sum_i\lvert w_i\rvert\right) = 2\mu_0\epsilon .
$$

Since $\epsilon$ was arbitrary, $Q_n[f]\to I[f]$ for **every continuous** $f$ — **no smoothness required**. $\blacksquare$

**Part 4 — the contrast with Newton–Cotes.** The final step used $\sum_i\lvert w_i\rvert = \mu_0$, which holds only for positive weights. For closed Newton–Cotes rules the weights turn negative at $n = 8$ and $\sum_i\lvert w_i\rvert$ grows geometrically:

| $n$ | 7 | 8 | 10 | 12 | 14 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| $\sum_i\lvert w_i\rvert$ (normalized) | $1.000$ | $1.451$ | $3.065$ | $7.532$ | $20.344$ |

Since $\sum\lvert w_i\rvert\to\infty$, the Banach–Steinhaus (uniform boundedness) principle applies: the family $\{Q_n\}$ of functionals on $C[a,b]$ is unbounded in norm, so there **exists** a continuous $f$ for which $Q_n[f]\not\to I[f]$ — this is the Pólya–Steklov theorem, and $1/(1+25x^{2})$ is an explicit witness. Simultaneously, the noise amplification $\sum\lvert w_i\rvert\,\delta$ diverges, so even where the rule converges in exact arithmetic it is useless in floating point.

**Where else positivity appears.** Clenshaw–Curtis, Gauss–Kronrod, Gauss–Lobatto, Gauss–Radau, and all composite rules built from midpoint/trapezoid/Simpson have positive weights. Every quadrature rule in production use does. Positivity is not a curiosity; it is the admission ticket.

$$
\boxed{w_i = \int L_i^{2}\rho \gt 0 \Rightarrow \textstyle\sum\lvert w_i\rvert = \mu_0 \Rightarrow \text{stability and convergence for all } f\in C[a,b]}
$$

*Key takeaway:* One inequality, $w_i \gt 0$, simultaneously delivers numerical stability and convergence for every continuous integrand — and its failure is exactly what kills high-order Newton–Cotes.

### Problem L3.4: The Monte Carlo rate, the crossover dimension, and the zero-variance proposal

Prove the $O(N^{-1/2})$ Monte Carlo rate with a confidence interval, derive the dimension at which it overtakes a tensor rule, and find the importance-sampling proposal that minimizes variance. Explain why the optimum is unattainable and what is done instead.

**Solution.**

**Part 1 — the rate and a confidence interval.** With $X_1,\ldots,X_N$ iid uniform on $\Omega$ and $\hat I_N = \frac{\lvert\Omega\rvert}{N}\sum_k f(X_k)$:

$$
\mathbb{E}[\hat I_N] = \lvert\Omega\rvert\,\mathbb{E}[f(X)] = \lvert\Omega\rvert\cdot\frac{1}{\lvert\Omega\rvert}\int_\Omega f = I \quad\text{(unbiased)},
$$

$$
\operatorname{Var}(\hat I_N) = \frac{\lvert\Omega\rvert^{2}}{N^{2}}\sum_{k=1}^{N}\operatorname{Var}\bigl(f(X_k)\bigr) = \frac{\lvert\Omega\rvert^{2}\sigma^{2}}{N},
$$

the cross terms vanishing by independence. Hence $\mathrm{RMSE} = \lvert\Omega\rvert\sigma N^{-1/2}$. By the central limit theorem $\sqrt N(\hat I_N - I)/(\lvert\Omega\rvert\sigma)\Rightarrow\mathcal N(0,1)$, so an asymptotic $95\%$ interval is

$$
\hat I_N \pm 1.96\,\frac{\lvert\Omega\rvert\hat\sigma}{\sqrt N}, \qquad \hat\sigma^{2} = \frac{1}{N-1}\sum_k\bigl(f(X_k)-\bar f\bigr)^{2}.
$$

**This error bar is free and computed from the same samples** — a practical advantage no deterministic rule offers. $\blacksquare$

**Part 2 — the crossover.** A tensor rule of order $k$ with $m$ points per axis costs $N = m^{d}$ and has error $O(m^{-k}) = O(N^{-k/d})$. Equating exponents with $N^{-1/2}$:

$$
\frac{k}{d} = \frac12 \implies d^{\ast} = 2k .
$$

Simpson ($k=4$) is overtaken at $d = 8$; trapezoid ($k=2$) at $d = 4$. Beyond that the deterministic rate collapses — at $d = 100$, tensor Simpson gives $N^{-0.04}$, while $3^{100} = 5.15\times10^{47}$ points are needed even to place a single node per axis triple.

**Part 3 — the optimal importance-sampling proposal.** Write $I = \int f = \int \frac{f(x)}{q(x)}q(x)dx = \mathbb{E}_q\!\left[\frac{f}{q}\right]$ for any density $q \gt 0$ on the support of $f$. The estimator $\hat I = \frac1N\sum_k \frac{f(X_k)}{q(X_k)}$, $X_k\sim q$, is unbiased with

$$
\operatorname{Var}_q(\hat I) = \frac{1}{N}\left( \int \frac{f(x)^{2}}{q(x)}dx - I^{2} \right).
$$

Minimize $\int f^{2}/q$ subject to $\int q = 1$. With a Lagrange multiplier $\lambda$, the stationarity condition $-\frac{f^{2}}{q^{2}} + \lambda = 0$ gives $q \propto \lvert f\rvert$. Rigorously, Cauchy–Schwarz gives

$$
\left(\int \lvert f\rvert\right)^{2} = \left( \int \frac{\lvert f\rvert}{\sqrt q}\sqrt q \right)^{2} \le \int\frac{f^{2}}{q}\int q = \int\frac{f^{2}}{q},
$$

with equality iff $\frac{\lvert f\rvert}{\sqrt q}\propto\sqrt q$, i.e. $q^{\ast} = \frac{\lvert f\rvert}{\int\lvert f\rvert}$. For $f \ge 0$ this gives

$$
\operatorname{Var}_{q^{\ast}}(\hat I) = \frac{1}{N}\left( \left(\int f\right)^{2} - I^{2} \right) = 0 .
$$

**Zero variance from a single sample.** $\blacksquare$

**Part 4 — why it is unattainable, and what is done instead.** The optimal proposal requires the normalizing constant $\int\lvert f\rvert$ — which for $f \ge 0$ *is the answer we are trying to compute*. Knowing $q^{\ast}$ means already knowing $I$. So the theorem is a design principle, not an algorithm:

- **Sample where $\lvert f\rvert$ is large.** Choose a tractable $q$ that mimics the shape of $\lvert f\rvert$ — a Gaussian centred at the mode (a Laplace approximation), a mixture, or a learned flow.
- **Guard the tails.** The variance $\int f^{2}/q$ is infinite if $q$ has lighter tails than $f^{2}$; a "good-looking" proposal that is slightly too narrow gives an estimator with infinite variance and deceptively stable-looking output. Diagnose with the effective sample size $\mathrm{ESS} = \frac{(\sum_k \omega_k)^{2}}{\sum_k \omega_k^{2}}$ and the Pareto-$\hat k$ statistic ($\hat k \lt 0.7$ is the usual threshold).
- **Learn $q$.** Adaptive importance sampling, cross-entropy methods, and normalizing-flow proposals iteratively fit $q$ toward $\lvert f\rvert$ — exactly the variational-inference objective, which is why VI and importance sampling are two views of the same problem.
- **Self-normalize.** When $f$ is known only up to a constant (unnormalized posteriors), use $\hat I = \frac{\sum_k \omega_k g(X_k)}{\sum_k \omega_k}$ with $\omega_k = \tilde p(X_k)/q(X_k)$ — biased at $O(1/N)$ but consistent, and the only option in Bayesian practice.

$$
\boxed{\mathrm{RMSE} = \sigma\lvert\Omega\rvert N^{-1/2};\ d^{\ast} = 2k;\ q^{\ast}\propto\lvert f\rvert \text{ gives zero variance but needs } I}
$$

*Key takeaway:* Monte Carlo's rate is fixed by the CLT and cannot be improved — but its *constant* can be driven arbitrarily low by matching the proposal to the integrand, which is the mathematical core of importance sampling, variational inference, and MCMC design.